In [1]:
import ee
import geemap

In [2]:
# authenticate and initialize Earth Engine

ee.Authenticate()
ee.Initialize(project="gsapp-map")

dataset = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")

center = [34.80251148191422, 31.6008201894651]
point = ee.Geometry.Point(center[0], center[1])
# image2024 = dataset.filterDate("2024-01-01", "2025-01-01").filterBounds(region).mosaic()

region = point.buffer(500000)

image2017 = dataset.filterDate("2017-01-01", "2018-01-01").filterBounds(region).mosaic()
image2022 = dataset.filterDate("2022-01-01", "2023-01-01").filterBounds(region).mosaic()
image2024 = dataset.filterDate("2024-01-01", "2025-01-01").filterBounds(region).mosaic()

# Visualize three axes of the embedding space as an RGB.
visParams = {min: -0.3, max: 0.3, "bands": ["A44", "A18", "A50"]}

Map = geemap.Map(center=center, zoom=4)
Map.add_basemap("TopPlusOpen.Grey")

Map.addLayer(image2017, visParams, "2017 embeddings")
Map.addLayer(image2024, visParams, "2024 embeddings")

Map.centerObject(point, zoom=6)
Map.setOptions("SATELLITE")
Map

Map(center=[31.6008201894651, 34.80251148191422], controls=(WidgetControl(options=['position', 'transparent_bg…

In [3]:
# This section is unrelated to the rest of the notebook
# Visualize three axes of the embedding space as an RGB (for export or display in Colab)

# Sample points for training
n_samples = 1000
training2024 = image2024.sample(
    region=region, scale=10, numPixels=n_samples, seed=100, geometries=True
)
training2017 = image2017.sample(
    region=region, scale=10, numPixels=n_samples, seed=100, geometries=True
)


# Function to train a model for desired number of clusters
def get_clusters(training, n_clusters):
    clusterer = ee.Clusterer.wekaKMeans(n_clusters).train(training)
    clustered = image2024.cluster(clusterer)
    return clustered


cluster10_2024 = get_clusters(training2024, 10)
cluster20_2024 = get_clusters(training2024, 20)
clusters40_2024 = get_clusters(training2024, 40)

cluster10_2017 = get_clusters(training2017, 10)
cluster20_2017 = get_clusters(training2017, 20)
clusters40_2017 = get_clusters(training2017, 40)

Map = geemap.Map(center=center, zoom=6)
Map.add_basemap("TopPlusOpen.Grey")

# Add clusters
Map.addLayer(cluster10_2024.clip(region).randomVisualizer(), {}, "2024: 10 Clusters")
Map.addLayer(cluster20_2024.clip(region).randomVisualizer(), {}, "2024: 20 Clusters")
Map.addLayer(clusters40_2024.clip(region).randomVisualizer(), {}, "2024: 40 Clusters")

Map.addLayer(cluster10_2017.clip(region).randomVisualizer(), {}, "2017: 10 Clusters")
Map.addLayer(cluster20_2017.clip(region).randomVisualizer(), {}, "2017: 20 Clusters")
Map.addLayer(clusters40_2017.clip(region).randomVisualizer(), {}, "2017: 40 Clusters")

Map

Map(center=[34.80251148191422, 31.6008201894651], controls=(WidgetControl(options=['position', 'transparent_bg…

In [4]:
# Compute per-pixel Euclidean distance between embedding vectors
bands = list(
    set(image2022.bandNames().getInfo()) & set(image2024.bandNames().getInfo())
)
diff = image2024.select(bands).subtract(image2022.select(bands))
sq = diff.pow(2)
sum_sq = sq.reduce(ee.Reducer.sum())
euclidean_dist = sum_sq.sqrt().rename("euclidean_dist")

Map = geemap.Map(center=center, zoom=6)
Map.add_basemap("TopPlusOpen.Grey")

# Visualize the Euclidean distance (difference magnitude)
Map.addLayer(
    euclidean_dist,
    {"min": 0, "max": 1, "palette": ["black", "white"]},
    "Embedding Difference (Euclidean)",
)

Map.centerObject(point, zoom=9)
Map

Map(center=[31.6008201894651, 34.80251148191422], controls=(WidgetControl(options=['position', 'transparent_bg…

In [6]:
# Color pixels by their minimum embedding distance to two reference points, for all available years

# Reference points
# Point 1: 31°14'24.55"N 34°17'29.17"E
ref1_lon = 34 + 17 / 60 + 29.17 / 3600
ref1_lat = 31 + 14 / 60 + 24.55 / 3600
ref_point1 = ee.Geometry.Point(ref1_lon, ref1_lat)

# Point 2: 31°32'47.32"N 34°33'33.14"E
ref2_lon = 34 + 33 / 60 + 33.14 / 3600
ref2_lat = 31 + 32 / 60 + 47.32 / 3600
ref_point2 = ee.Geometry.Point(ref2_lon, ref2_lat)

# Center between the two points
center_lon = (ref1_lon + ref2_lon) / 2
center_lat = (ref1_lat + ref2_lat) / 2
center_point = ee.Geometry.Point(center_lon, center_lat)

# Region with ~40 km radius
region_40km = center_point.buffer(40000)

# Years available in the annual AlphaEarth embedding dataset
year_list = (
    ee.List(dataset.aggregate_array("system:time_start"))
    .map(lambda t: ee.Date(t).get("year"))
    .distinct()
    .sort()
    .getInfo()
)

Map = geemap.Map(center=[center_lon, center_lat], zoom=10)
Map.add_basemap("TopPlusOpen.Grey")
Map.setOptions("SATELLITE")

dist_vis = {
    "min": 0,
    "max": 1,
    "palette": ["0d0887", "7e03a8", "cc4778", "f89540", "f0f921"],
}

for year in year_list:
    yearly_img = (
        dataset.filterDate(f"{year}-01-01", f"{year + 1}-01-01")
        .filterBounds(region_40km)
        .mosaic()
    )

    bands = yearly_img.bandNames()
    
    # Get embeddings at both reference points
    ref1_dict = yearly_img.reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=ref_point1,
        scale=10,
        bestEffort=True,
        maxPixels=1e8,
    )
    
    ref2_dict = yearly_img.reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=ref_point2,
        scale=10,
        bestEffort=True,
        maxPixels=1e8,
    )

    ref1_values = ee.List(bands.map(lambda b: ref1_dict.get(ee.String(b))))
    ref1_embedding = ee.Image.constant(ref1_values).rename(bands)
    
    ref2_values = ee.List(bands.map(lambda b: ref2_dict.get(ee.String(b))))
    ref2_embedding = ee.Image.constant(ref2_values).rename(bands)

    # Calculate distance to each reference point
    dist1 = (
        yearly_img.select(bands)
        .subtract(ref1_embedding)
        .pow(2)
        .reduce(ee.Reducer.sum())
        .sqrt()
    )
    
    dist2 = (
        yearly_img.select(bands)
        .subtract(ref2_embedding)
        .pow(2)
        .reduce(ee.Reducer.sum())
        .sqrt()
    )
    
    # Use minimum distance
    min_dist = dist1.min(dist2).rename("embedding_dist")

    Map.addLayer(min_dist.clip(region_40km), dist_vis, f"{year} min distance")

Map.addLayer(ref_point1, {"color": "cyan"}, "Reference point 1")
Map.addLayer(ref_point2, {"color": "magenta"}, "Reference point 2")
Map.centerObject(center_point, zoom=10)
Map

Map(center=[31.393315277777774, 34.42532083333333], controls=(WidgetControl(options=['position', 'transparent_…

In [9]:
# Same analysis using raw Landsat imagery instead of embeddings
# Using spectral distance to classify pixels based on similarity to reference points

# Merge multiple Landsat missions for full time coverage
landsat5 = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2")  # 1984-2012
landsat7 = ee.ImageCollection("LANDSAT/LE07/C02/T1_L2")  # 1999-present
landsat8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")  # 2013-present
landsat9 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")  # 2021-present

# Optical bands for spectral analysis (consistent across missions)
optical_bands = ["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B7"]

# Function to apply scaling and cloud masking
def prep_landsat(image):
    # Apply scaling factors
    optical = image.select(optical_bands).multiply(0.0000275).add(-0.2)
    # Basic cloud mask using QA band
    qa = image.select('QA_PIXEL')
    cloud_mask = qa.bitwiseAnd(1 << 3).eq(0)  # Bit 3: Cloud
    return optical.updateMask(cloud_mask).copyProperties(image, ["system:time_start"])

# Merge all Landsat collections
landsat_all = landsat5.merge(landsat7).merge(landsat8).merge(landsat9)

# 5-year increments from 1985 to 2025
year_list_landsat = list(range(1985, 2026, 5))

Map = geemap.Map(center=[center_lon, center_lat], zoom=10)
Map.add_basemap("TopPlusOpen.Grey")
Map.setOptions("SATELLITE")

dist_vis_spectral = {"min": 0, "max": 0.3, "palette": ["0d0887", "7e03a8", "cc4778", "f89540", "f0f921"]}

for year in year_list_landsat:
    # Get median composite for the year (2-year window for better coverage)
    yearly_landsat = (
        landsat_all.filterDate(f"{year}-01-01", f"{year + 2}-01-01")
        .filterBounds(region_40km)
        .map(prep_landsat)
        .median()
    )
    
    # Get reference spectra from this year's imagery
    ref1_spectrum = yearly_landsat.reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=ref_point1,
        scale=30,
    )
    
    ref2_spectrum = yearly_landsat.reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=ref_point2,
        scale=30,
    )
    
    # Create constant images from reference spectra
    ref1_bands = ee.List(optical_bands)
    ref1_values = ref1_bands.map(lambda b: ref1_spectrum.get(ee.String(b)))
    ref1_img = ee.Image.constant(ref1_values).rename(optical_bands)
    
    ref2_values = ref1_bands.map(lambda b: ref2_spectrum.get(ee.String(b)))
    ref2_img = ee.Image.constant(ref2_values).rename(optical_bands)
    
    # Euclidean distance in spectral space
    dist1_spectral = (
        yearly_landsat.subtract(ref1_img)
        .pow(2)
        .reduce(ee.Reducer.sum())
        .sqrt()
    )
    
    dist2_spectral = (
        yearly_landsat.subtract(ref2_img)
        .pow(2)
        .reduce(ee.Reducer.sum())
        .sqrt()
    )
    
    # Use minimum distance
    min_dist_spectral = dist1_spectral.min(dist2_spectral).rename("spectral_dist")
    
    Map.addLayer(min_dist_spectral.clip(region_40km), dist_vis_spectral, f"{year} Spectral Distance", False)

# Show the most recent year by default
Map.addLayer(
    landsat_all.filterDate("2023-01-01", "2024-01-01")
    .filterBounds(region_40km)
    .map(prep_landsat)
    .median()
    .clip(region_40km),
    {"min": 0, "max": 0.3, "bands": ["SR_B4", "SR_B3", "SR_B2"]},
    "2023 True Color"
)

Map.addLayer(ref_point1, {"color": "cyan"}, "Reference point 1")
Map.addLayer(ref_point2, {"color": "magenta"}, "Reference point 2")
Map.centerObject(center_point, zoom=10)

# Commented out Spectral Angle Mapper (SAM) - alternative method
# def spectral_angle(image, reference):
#     dot_product = image.multiply(reference).reduce(ee.Reducer.sum())
#     image_norm = image.pow(2).reduce(ee.Reducer.sum()).sqrt()
#     ref_norm = reference.pow(2).reduce(ee.Reducer.sum()).sqrt()
#     cos_angle = dot_product.divide(image_norm.multiply(ref_norm))
#     angle = cos_angle.acos()  # angle in radians
#     return angle

Map

Map(center=[31.393315277777774, 34.42532083333333], controls=(WidgetControl(options=['position', 'transparent_…